# 08_3_7 DQA Scene Phase2 R-SCoLQ Policy

`02_train_and_validate_round_stable_scolq.ipynb` で作った **R-SCoLQ** を Phase2 に入れる実験。

08_3_6の問題は、roundを回すほど `pseudo_total` と `mean_scolq` が一緒に増え、SCoLQ上は良く見えるのにmAPが落ちたこと。08_3_7では、sourceで学習したstatic bbox qualityに加えて、前roundのpseudo statsから `global_multiplier` と `class_multiplier` を作り、pseudo label inflationを次roundで罰する。

```text
R-SCoLQ score = static bbox quality * round global multiplier * class multiplier
```

このpolicyは target GT checkpoint選択ではなく、学習中に観測できるpseudo statsだけで自己制限する。

In [1]:
from __future__ import annotations

import json
import os
import signal
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "dynamic_quality_aware_classwise_aggregation").exists() and (path / "navigating_data_heterogeneity").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_rscolq_policy.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
RSCOLQ_ROOT = DQA_ROOT / "source_calibrated_localization_quality"
RSCOLQ_MODEL = RSCOLQ_ROOT / "artifacts" / "rscolq_best.joblib"
RSCOLQ_RANKING = RSCOLQ_ROOT / "reports" / "rscolq_model_ranking.csv"
RSCOLQ_DIAG = RSCOLQ_ROOT / "reports" / "rscolq_0836_round_diagnostics.csv"
BASE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_7_phase2_rscolq_policy"
BASE_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_7_phase2_rscolq_policy"
BASE_LOG_ROOT = DQA_ROOT / "logs_dqa08_3_7_phase2_rscolq_policy"

for path in [RUN_SCRIPT, EVAL_SCRIPT, SOURCE_WORK_ROOT, RSCOLQ_MODEL]:
    print(path, "exists=", path.exists())

BASE_WORK_ROOT.mkdir(parents=True, exist_ok=True)
BASE_STATS_ROOT.mkdir(parents=True, exist_ok=True)
BASE_LOG_ROOT.mkdir(parents=True, exist_ok=True)

if RSCOLQ_RANKING.exists():
    ranking = pd.read_csv(RSCOLQ_RANKING)
    display(ranking[["candidate", "n_features", "ap50_quality", "ap75_quality", "p50_at_10pct", "proxy_penalty", "selection_score"]])
if RSCOLQ_DIAG.exists():
    diag = pd.read_csv(RSCOLQ_DIAG)
    display(diag[["round", "pseudo_total", "mean_scolq", "global_multiplier", "mean_class_multiplier", "rscolq_proxy", "map50", "map50_95"]])

print("workspace:", BASE_WORK_ROOT)
print("stats:", BASE_STATS_ROOT)
print("logs:", BASE_LOG_ROOT)

/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_rscolq_policy.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/rscolq_best.joblib exists= True


,candidate,n_features,ap50_quality,ap75_quality,p50_at_10pct,proxy_penalty,selection_score
0,low_conf:hgb,19,0.767833,0.795002,0.653509,0.045,2.480720
1,rank_conf:hgb,21,0.768683,0.794828,0.654162,0.145,2.382401
2,all_proxy:hgb,24,0.768551,0.795472,0.653742,0.295,2.231998
3,safe_no_conf:hgb,18,0.614495,0.512432,0.547675,0.030,1.930348


,round,pseudo_total,mean_scolq,global_multiplier,mean_class_multiplier,rscolq_proxy,map50,map50_95
0,1,770515.0,0.443193,1.000000,0.997977,0.442297,0.382,0.211
1,2,773621.0,0.448643,1.000000,0.996683,0.447155,0.381,0.212
2,3,757997.0,0.450761,1.000000,0.999152,0.450378,0.377,0.211
3,4,760731.0,0.453214,0.977300,0.999490,0.442700,NaN,NaN
4,5,770963.0,0.456014,0.950322,0.998685,0.432790,NaN,NaN
5,6,785763.0,0.460806,0.885269,0.995085,0.405932,NaN,NaN
6,7,795512.0,0.466490,0.805568,0.990184,0.372101,NaN,NaN
7,8,807088.0,0.470166,0.742671,0.987176,0.344701,NaN,NaN
8,9,817731.0,0.476864,0.666687,0.980792,0.311813,NaN,NaN
9,10,819653.0,0.487045,0.597721,0.956666,0.278502,0.357,0.196


workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_7_phase2_rscolq_policy
stats: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_7_phase2_rscolq_policy
logs: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_7_phase2_rscolq_policy


## Variant Design

まずは `a_rscolq_round_stable_r003` を本命として回す。

- A: 08_3_6 Aに近い設定。ただしR-SCoLQ multiplierで後半のpseudo label膨張を抑える。
- B: high gateを少し上げるprecision寄り。
- C: bbox lossをround後半でさらに弱める。R-SCoLQが効いてもbbox regressionそのものが危険な場合の対照。

最初はAだけ。Aがr010まで崩れにくいか、少なくともr001/r002の良さを後半で破壊しにくいかを見る。

In [2]:
PHASE2_ROUNDS_PER_VARIANT = 10
BATCH_SIZE = 160
WORKERS = 10
GPUS = 2
MASTER_PORT_BASE = 30020
STREAM_TRAIN_OUTPUT = False
MIN_FREE_GIB = 30

SELECTED_VARIANTS: list[str] = ["a_rscolq_round_stable_r003"]

VARIANTS = [
    {
        "name": "a_rscolq_round_stable_r003",
        "description": "R-SCoLQ本命。08_3_6 Aに近いgateで始め、前round statsに基づくglobal/class multiplierで後半のpseudo膨張を罰する。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "classwise_blend": 0.065,
        "server_anchor": 13.0,
        "temperature": 2.7,
        "stability_lambda": 0.66,
        "residual_start": 0.14,
        "residual_end": 0.06,
        "min_server_alpha_start": 0.70,
        "min_server_alpha_end": 0.74,
        "env": {
            "DQA0837_RSCOLQ_LOW": 0.10,
            "DQA0837_RSCOLQ_MID": 0.30,
            "DQA0837_RSCOLQ_HIGH": 0.60,
            "DQA0837_NMS_CONF_THRES": 0.01,
            "DQA0837_TEACHER_LOSS_WEIGHT": 0.32,
            "DQA0837_BOX_LOSS_WEIGHT_START": 0.010,
            "DQA0837_BOX_LOSS_WEIGHT_END": 0.006,
            "DQA0837_OBJ_LOSS_WEIGHT": 0.32,
            "DQA0837_CLS_LOSS_WEIGHT": 0.08,
            "DQA0837_LOW_MID_OBJ_WEIGHT": 0.25,
            "DQA0837_MID_HIGH_OBJ_WEIGHT": 0.70,
            "DQA0837_CLIENT_LR0_START": 0.0010,
            "DQA0837_CLIENT_LR0_END": 0.00022,
            "DQA0837_SERVER_LR0_START": 0.0030,
            "DQA0837_SERVER_LR0_END": 0.0010,
            "DQA0837_MAX_PSEUDO_PER_IMAGE": 80,
            "DQA0837_MAX_PSEUDO_PER_CLASS_IMAGE": 25,
        },
    },
    {
        "name": "b_rscolq_strict_gate_r003",
        "description": "R-SCoLQ + strict gate。static scoreが低めに出る場合に備えつつ、bbox positiveはさらに絞る。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "classwise_blend": 0.055,
        "server_anchor": 15.0,
        "temperature": 2.9,
        "stability_lambda": 0.70,
        "residual_start": 0.11,
        "residual_end": 0.04,
        "min_server_alpha_start": 0.74,
        "min_server_alpha_end": 0.80,
        "env": {
            "DQA0837_RSCOLQ_LOW": 0.12,
            "DQA0837_RSCOLQ_MID": 0.35,
            "DQA0837_RSCOLQ_HIGH": 0.65,
            "DQA0837_NMS_CONF_THRES": 0.012,
            "DQA0837_TEACHER_LOSS_WEIGHT": 0.28,
            "DQA0837_BOX_LOSS_WEIGHT_START": 0.008,
            "DQA0837_BOX_LOSS_WEIGHT_END": 0.004,
            "DQA0837_OBJ_LOSS_WEIGHT": 0.30,
            "DQA0837_CLS_LOSS_WEIGHT": 0.06,
            "DQA0837_LOW_MID_OBJ_WEIGHT": 0.20,
            "DQA0837_MID_HIGH_OBJ_WEIGHT": 0.60,
            "DQA0837_CLIENT_LR0_START": 0.0008,
            "DQA0837_CLIENT_LR0_END": 0.00018,
            "DQA0837_SERVER_LR0_START": 0.0024,
            "DQA0837_SERVER_LR0_END": 0.0008,
            "DQA0837_MAX_PSEUDO_PER_IMAGE": 65,
            "DQA0837_MAX_PSEUDO_PER_CLASS_IMAGE": 18,
        },
    },
    {
        "name": "c_rscolq_bbox_decay_r003",
        "description": "R-SCoLQ + bbox loss decay。pseudo score制御だけでなくbbox regressionの後半影響をさらに落とす。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "neck_head",
        "classwise_blend": 0.050,
        "server_anchor": 16.0,
        "temperature": 3.0,
        "stability_lambda": 0.72,
        "residual_start": 0.10,
        "residual_end": 0.03,
        "min_server_alpha_start": 0.76,
        "min_server_alpha_end": 0.84,
        "env": {
            "DQA0837_RSCOLQ_LOW": 0.10,
            "DQA0837_RSCOLQ_MID": 0.30,
            "DQA0837_RSCOLQ_HIGH": 0.60,
            "DQA0837_NMS_CONF_THRES": 0.01,
            "DQA0837_TEACHER_LOSS_WEIGHT": 0.25,
            "DQA0837_BOX_LOSS_WEIGHT_START": 0.008,
            "DQA0837_BOX_LOSS_WEIGHT_END": 0.001,
            "DQA0837_OBJ_LOSS_WEIGHT": 0.28,
            "DQA0837_CLS_LOSS_WEIGHT": 0.05,
            "DQA0837_LOW_MID_OBJ_WEIGHT": 0.18,
            "DQA0837_MID_HIGH_OBJ_WEIGHT": 0.55,
            "DQA0837_CLIENT_LR0_START": 0.00075,
            "DQA0837_CLIENT_LR0_END": 0.00012,
            "DQA0837_SERVER_LR0_START": 0.0022,
            "DQA0837_SERVER_LR0_END": 0.0007,
            "DQA0837_MAX_PSEUDO_PER_IMAGE": 70,
            "DQA0837_MAX_PSEUDO_PER_CLASS_IMAGE": 20,
        },
    },
]

selected = [v for v in VARIANTS if (not SELECTED_VARIANTS or v["name"] in SELECTED_VARIANTS)]
print("selected:", [v["name"] for v in selected])

selected: ['a_rscolq_round_stable_r003']


In [3]:
def variant_work_root(variant: dict) -> Path:
    return BASE_WORK_ROOT / variant["name"]


def variant_stats_root(variant: dict) -> Path:
    return BASE_STATS_ROOT / variant["name"]


def variant_runner_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_runner.out"


def variant_train_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_train.log"


def variant_pid_path(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}.pid"


def variant_env(variant: dict) -> dict[str, str]:
    env = os.environ.copy()
    stats_root = variant_stats_root(variant)
    stats_root.mkdir(parents=True, exist_ok=True)
    env["DQA0837_VARIANT"] = variant["name"]
    env["DQA0837_SOURCE_WORK_ROOT"] = str(SOURCE_WORK_ROOT)
    env["DQA0837_SOURCE_PHASE1_ROUND"] = str(variant["source_phase1_round"])
    env["DQA0837_RSCOLQ_MODEL"] = str(RSCOLQ_MODEL)
    env["DQA0837_CLIENT_TRAIN_SCOPE"] = variant["client_train_scope"]
    env["DQA0837_SERVER_TRAIN_SCOPE"] = "all"
    env["DQA0837_RESIDUAL_START"] = str(variant["residual_start"])
    env["DQA0837_RESIDUAL_END"] = str(variant["residual_end"])
    env["DQA0837_MIN_SERVER_ALPHA_START"] = str(variant["min_server_alpha_start"])
    env["DQA0837_MIN_SERVER_ALPHA_END"] = str(variant["min_server_alpha_end"])
    env["DQA0837_PHASE2_ROUNDS"] = str(PHASE2_ROUNDS_PER_VARIANT)
    env["DQA08_STATS_ROOT"] = str(stats_root.resolve())
    env["DQA08_THRESHOLD_LOG"] = str((stats_root / "phase2_rscolq_policy_schedule.jsonl").resolve())
    env["DQA_STATS_QUALITY_MODE"] = "rscolq"
    env["DQA0834_STATS_QUALITY_MODE"] = "rscolq"
    env.setdefault("ET_SKIP_AFTER_TRAIN_BEST_VAL", "1")
    for key, value in variant.get("env", {}).items():
        env[key] = str(value)
    return env


def train_cmd(variant: dict, *, stream: bool = STREAM_TRAIN_OUTPUT) -> list[str]:
    cmd = [
        sys.executable,
        str(RUN_SCRIPT),
        "--workspace-root", str(variant_work_root(variant)),
        "--stats-root", str(variant_stats_root(variant)),
        "--phase1-rounds", "0",
        "--phase2-rounds", str(PHASE2_ROUNDS_PER_VARIANT),
        "--batch-size", str(BATCH_SIZE),
        "--workers", str(WORKERS),
        "--gpus", str(GPUS),
        "--master-port", str(MASTER_PORT_BASE + selected.index(variant)),
        "--log-file", str(variant_train_log(variant)),
        "--classwise-blend", str(variant["classwise_blend"]),
        "--server-anchor", str(variant["server_anchor"]),
        "--temperature", str(variant["temperature"]),
        "--stability-lambda", str(variant["stability_lambda"]),
        "--dqa-start-phase", str(variant["dqa_start_phase"]),
    ]
    if stream:
        cmd.append("--stream-train-output")
    return cmd


def history_rows(variant: dict) -> list[dict]:
    path = variant_work_root(variant) / "history.json"
    if not path.exists():
        return []
    return json.loads(path.read_text())


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text().strip())
    except Exception:
        return None


def pid_state(pid: int | None) -> str:
    if not pid:
        return "missing"
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return "stopped"
    except PermissionError:
        return "unknown"
    return "running"


def tail_lines(path: Path, n: int = 40) -> list[str]:
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()[-n:]

In [4]:
# 実行セル。途中で止まっても history/global checkpoint があれば再利用します。
if not selected:
    raise RuntimeError("No selected variants")
if not RSCOLQ_MODEL.exists():
    raise FileNotFoundError(f"Missing R-SCoLQ model: {RSCOLQ_MODEL}")

for variant in selected:
    history = history_rows(variant)
    completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
    print("\n===", variant["name"], "===")
    print(variant["description"])
    print(f"completed_phase2: {completed_phase2}/{PHASE2_ROUNDS_PER_VARIANT}")
    if completed_phase2 >= PHASE2_ROUNDS_PER_VARIANT:
        print("already completed")
        continue

    pid_path = variant_pid_path(variant)
    existing_pid = read_pid(pid_path)
    if pid_state(existing_pid) == "running":
        print("already running pid:", existing_pid)
        continue

    runner_log = variant_runner_log(variant)
    runner_log.parent.mkdir(parents=True, exist_ok=True)
    cmd = train_cmd(variant)
    env = variant_env(variant)
    print("cmd:", " ".join(cmd))
    print("runner_log:", runner_log)
    print("train_log:", variant_train_log(variant))

    with runner_log.open("a", encoding="utf-8", buffering=1) as log:
        log.write("\n" + "=" * 100 + "\n")
        log.write(" ".join(cmd) + "\n")
        process = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        pid_path.write_text(str(process.pid), encoding="utf-8")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        rc = process.wait()
        if rc != 0:
            raise RuntimeError(f"{variant['name']} failed with exit code {rc}. See {runner_log}")
        print("variant completed:", variant["name"])


=== a_rscolq_round_stable_r003 ===
R-SCoLQ本命。08_3_6 Aに近いgateで始め、前round statsに基づくglobal/class multiplierで後半のpseudo膨張を罰する。
completed_phase2: 0/10
cmd: /root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_rscolq_policy.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_7_phase2_rscolq_policy/a_rscolq_round_stable_r003 --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_7_phase2_rscolq_policy/a_rscolq_round_stable_r003 --phase1-rounds 0 --phase2-rounds 10 --batch-size 160 --workers 10 --gpus 2 --master-port 30020 --log-file /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_7_phase2_rscolq_policy/a_rscolq_round_stable_r003_train.log --classwise-blend 0.065 --server-anchor 13.0 --temperature 2.7 --stability-lambda 0.66 --dqa-start-phase 2
runner_log: /app/Object_Detection/d

In [5]:
# 進捗確認セル
rows = []
for variant in selected:
    history = history_rows(variant)
    latest = Path(history[-1]["global"]) if history else variant_work_root(variant) / "global_checkpoints" / "round000_warmup.pt"
    rows.append({
        "variant": variant["name"],
        "pid": read_pid(variant_pid_path(variant)),
        "pid_state": pid_state(read_pid(variant_pid_path(variant))),
        "phase2": f"{sum(1 for row in history if int(row.get('phase', 0)) == 2)}/{PHASE2_ROUNDS_PER_VARIANT}",
        "latest": str(latest),
        "latest_exists": latest.exists(),
        "free_gib": round(shutil.disk_usage(variant_work_root(variant)).free / 1024**3, 2) if variant_work_root(variant).exists() else None,
    })

display(pd.DataFrame(rows))

for variant in selected:
    print("\n===", variant["name"], "runner tail ===")
    print("\n".join(tail_lines(variant_runner_log(variant), 24)))
    sched = variant_stats_root(variant) / "phase2_rscolq_policy_schedule.jsonl"
    if sched.exists():
        print("\n--- R-SCoLQ schedule tail ---")
        print("\n".join(tail_lines(sched, 12)))

,variant,pid,pid_state,phase2,latest,latest_exists,free_gib
0,a_rscolq_round_stable_r003,1578780,stopped,10/10,/app/Object_Detection/dynamic_quality_aware_cl...,True,80.61



=== a_rscolq_round_stable_r003 runner tail ===
Training output appended to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_7_phase2_rscolq_policy/a_rscolq_round_stable_r003_train.log
DQA08_3_7 R-SCoLQ gate: client=1 round=9 low=0.10 mid=0.30 high=0.60 global_mult=1.000 class_mult_mean=0.985 context=previous_round_stats model=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/rscolq_best.joblib
/root/micromamba/envs/al_yolov8/bin/python -m torch.distributed.run --nproc_per_node 2 --master_port 30020 train.py --cfg /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_7_phase2_rscolq_policy/a_rscolq_round_stable_r003/configs/runtime_phase2_client_round9_dqa_phase2_round009_client1_citystreet.yaml
Training output appended to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_7_phase2_rscolq_policy/a_rscolq_round_stable_r003_train.

In [6]:
# early/final checkpoint 評価。r001/r002/r003/r010 を見る。
EVAL_WORKSPACE = variant_work_root(selected[0])
REPORT_DIR = EVAL_WORKSPACE / "validation_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

checkpoints: list[tuple[str, Path]] = []
seed03 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round003_global.pt"
seed12 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round012_global.pt"
old0834_c02 = DQA_ROOT / "efficientteacher_dqa08_3_4_phase2_feature_quality_sweep" / "c_feature_balanced_neck_head_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0834_d02 = DQA_ROOT / "efficientteacher_dqa08_3_4_phase2_feature_quality_sweep" / "d_feature_conservative_min_gate_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0836_a01 = DQA_ROOT / "efficientteacher_dqa08_3_6_phase2_scolq_policy" / "a_scolq_soft_bbox_r003" / "global_checkpoints" / "phase2_round001_global.pt"
old0836_a02 = DQA_ROOT / "efficientteacher_dqa08_3_6_phase2_scolq_policy" / "a_scolq_soft_bbox_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0836_a10 = DQA_ROOT / "efficientteacher_dqa08_3_6_phase2_scolq_policy" / "a_scolq_soft_bbox_r003" / "global_checkpoints" / "phase2_round010_global.pt"
checkpoints.extend([
    ("p1_r003", seed03),
    ("p1_r012", seed12),
    ("old08_3_4_c_r002", old0834_c02),
    ("old08_3_4_d_r002", old0834_d02),
    ("old08_3_6_a_r001", old0836_a01),
    ("old08_3_6_a_r002", old0836_a02),
    ("old08_3_6_a_r010", old0836_a10),
])

for variant in selected:
    root = variant_work_root(variant) / "global_checkpoints"
    for r in [1, 2, 3, PHASE2_ROUNDS_PER_VARIANT]:
        ckpt = root / f"phase2_round{r:03d}_global.pt"
        if ckpt.exists():
            checkpoints.append((f"{variant['name']}_r{r:03d}", ckpt))

for label, path in checkpoints:
    print(label, path, "exists=", path.exists())

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace", str(EVAL_WORKSPACE),
    "--splits", "total",
    "--batch-size", "16",
    "--no-plots",
    "--verbose",
]
for label, path in checkpoints:
    if path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0837_early_total.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)
else:
    print("No summary yet:", summary_csv)

p1_r003 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt exists= True
p1_r012 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round012_global.pt exists= True
old08_3_4_c_r002 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_4_phase2_feature_quality_sweep/c_feature_balanced_neck_head_r003/global_checkpoints/phase2_round002_global.pt exists= True
old08_3_4_d_r002 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_4_phase2_feature_quality_sweep/d_feature_conservative_min_gate_r003/global_checkpoints/phase2_round002_global.pt exists= True
old08_3_6_a_r001 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003/global_checkpoints/p

,checkpoint_label,split,precision,recall,map50,map50_95
3,old08_3_4_d_r002,scene_total,0.503,0.399,0.381,0.213
5,old08_3_6_a_r002,scene_total,0.489,0.411,0.381,0.212
7,a_rscolq_round_stable_r003_r001,scene_total,0.485,0.406,0.381,0.212
8,a_rscolq_round_stable_r003_r002,scene_total,0.494,0.406,0.381,0.212
2,old08_3_4_c_r002,scene_total,0.492,0.408,0.380,0.212
4,old08_3_6_a_r001,scene_total,0.493,0.401,0.382,0.211
9,a_rscolq_round_stable_r003_r003,scene_total,0.503,0.400,0.377,0.211
0,p1_r003,scene_total,0.471,0.401,0.375,0.204
1,p1_r012,scene_total,0.743,0.326,0.368,0.199
6,old08_3_6_a_r010,scene_total,0.483,0.389,0.357,0.196


saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_7_phase2_rscolq_policy/a_rscolq_round_stable_r003/validation_reports/paper_protocol_eval_summary_0837_early_total.csv


In [7]:
# 4 split final evaluation。上の total 評価で良い checkpoint を BEST_LABELS に入れる。
BEST_LABELS = []

ckpt_map = {label: path for label, path in checkpoints if path.exists()}
if not BEST_LABELS:
    BEST_LABELS = [
        label for label in ckpt_map
        if label.endswith("_r001") or label.endswith("_r002") or label.endswith(f"r{PHASE2_ROUNDS_PER_VARIANT:03d}")
    ]

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace", str(EVAL_WORKSPACE),
    "--splits", "highway,citystreet,residential,total",
    "--batch-size", "16",
    "--no-plots",
    "--verbose",
]
for label in ["p1_r003", "p1_r012", "old08_3_4_c_r002", "old08_3_4_d_r002", "old08_3_6_a_r001", "old08_3_6_a_r002", "old08_3_6_a_r010", *BEST_LABELS]:
    path = ckpt_map.get(label)
    if path and path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0837_selected_4splits.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)

/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_7_phase2_rscolq_policy/a_rscolq_round_stable_r003 --splits highway,citystreet,residential,total --batch-size 16 --no-plots --verbose --checkpoint p1_r003=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt --checkpoint p1_r012=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round012_global.pt --checkpoint old08_3_4_c_r002=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_4_phase2_feature_quality_sweep/c_feature_balanced_neck_head_r003/global_checkpoints/phase2_round002_global.pt --checkpoint old08_3_4_d_r002=/app/

,checkpoint_label,split,precision,recall,map50,map50_95
13,old08_3_4_d_r002,citystreet,0.488,0.418,0.387,0.216
21,old08_3_6_a_r002,citystreet,0.488,0.418,0.387,0.216
33,old08_3_4_d_r002,citystreet,0.488,0.418,0.387,0.216
41,old08_3_6_a_r002,citystreet,0.488,0.418,0.387,0.216
53,a_rscolq_round_stable_r003_r002,citystreet,0.493,0.413,0.387,0.216
9,old08_3_4_c_r002,citystreet,0.493,0.414,0.386,0.216
29,old08_3_4_c_r002,citystreet,0.493,0.414,0.386,0.216
17,old08_3_6_a_r001,citystreet,0.484,0.414,0.387,0.215
37,old08_3_6_a_r001,citystreet,0.484,0.414,0.387,0.215
49,a_rscolq_round_stable_r003_r001,citystreet,0.485,0.413,0.387,0.215


saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_7_phase2_rscolq_policy/a_rscolq_round_stable_r003/validation_reports/paper_protocol_eval_summary_0837_selected_4splits.csv


In [8]:
# pseudoGT / R-SCoLQ quality / multiplier 推移を確認するセル。
rows = []
for variant in selected:
    stats_root = variant_stats_root(variant)
    for r in range(1, PHASE2_ROUNDS_PER_VARIANT + 1):
        path = stats_root / f"phase2_round{r:03d}.json"
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        counts = [0.0] * 10
        qsum = [0.0] * 10
        objsum = [0.0] * 10
        clssum = [0.0] * 10
        for client in data.get("clients", []):
            for i, value in enumerate(client.get("counts", [])[:10]):
                counts[i] += float(value)
            for i, value in enumerate(client.get("quality_sums", [])[:10]):
                qsum[i] += float(value)
            for i, value in enumerate(client.get("objectness_sums", [])[:10]):
                objsum[i] += float(value)
            for i, value in enumerate(client.get("class_confidence_sums", [])[:10]):
                clssum[i] += float(value)
        total = sum(counts)
        rows.append({
            "variant": variant["name"],
            "round": r,
            "pseudo_total": total,
            "mean_rscolq_quality": sum(qsum) / total if total else None,
            "mean_objectness": sum(objsum) / total if total else None,
            "mean_class_conf": sum(clssum) / total if total else None,
            "active_classes": sum(1 for x in counts if x > 0),
            "person_count": counts[0],
            "car_count": counts[2],
            "traffic_light_count": counts[7],
            "traffic_sign_count": counts[8],
        })

stats_df = pd.DataFrame(rows)
if not stats_df.empty:
    display(stats_df)
    display(stats_df.groupby("variant")[["pseudo_total", "mean_rscolq_quality", "mean_objectness", "active_classes"]].agg(["min", "max", "mean"]))
else:
    print("No pseudo stats yet")

for variant in selected:
    state_path = variant_work_root(variant) / "dqa_cwa_state.json"
    if state_path.exists():
        state = json.loads(state_path.read_text())
        gate = state.get("phase2_rscolq_policy", [])
        if gate:
            print("\n", variant["name"])
            display(pd.DataFrame(gate))
    sched = variant_stats_root(variant) / "phase2_rscolq_policy_schedule.jsonl"
    if sched.exists():
        sched_rows = [json.loads(line) for line in sched.read_text().splitlines() if line.strip()]
        if sched_rows:
            display(pd.DataFrame(sched_rows))

,variant,round,pseudo_total,mean_rscolq_quality,mean_objectness,mean_class_conf,active_classes,person_count,car_count,traffic_light_count,traffic_sign_count
0,a_rscolq_round_stable_r003,1,825251.0,0.647745,0.367722,0.881988,8,92517.0,316754.0,123671.0,255763.0
1,a_rscolq_round_stable_r003,2,720935.0,0.078157,0.248746,0.875411,8,67171.0,304640.0,91996.0,215204.0
2,a_rscolq_round_stable_r003,3,810637.0,0.641007,0.371161,0.889296,8,95475.0,317081.0,116925.0,245034.0
3,a_rscolq_round_stable_r003,4,710358.0,0.081023,0.260983,0.886881,8,64926.0,307391.0,94772.0,206798.0
4,a_rscolq_round_stable_r003,5,814331.0,0.648034,0.374294,0.890219,8,91183.0,316623.0,120514.0,250238.0
5,a_rscolq_round_stable_r003,6,727469.0,0.083843,0.279109,0.889599,8,67699.0,309574.0,98852.0,214544.0
6,a_rscolq_round_stable_r003,7,829204.0,0.659028,0.383180,0.890341,9,95651.0,317017.0,123904.0,256260.0
7,a_rscolq_round_stable_r003,8,756766.0,0.086688,0.302971,0.887442,8,70410.0,311492.0,105957.0,227935.0
8,a_rscolq_round_stable_r003,9,841382.0,0.666503,0.393394,0.887654,8,96636.0,317143.0,125833.0,261360.0
9,a_rscolq_round_stable_r003,10,785429.0,0.090328,0.337146,0.886977,8,83785.0,313545.0,108465.0,238513.0


pseudo_total                      \
                                    min       max      mean   
variant                                                       
a_rscolq_round_stable_r003     710358.0  841382.0  782176.2   

                           mean_rscolq_quality                      \
                                           min       max      mean   
variant                                                              
a_rscolq_round_stable_r003            0.078157  0.666503  0.368236   

                           mean_objectness                     active_classes  \
                                       min       max      mean            min   
variant                                                                         
a_rscolq_round_stable_r003        0.248746  0.393394  0.331871              8   

                                     
                           max mean  
variant                              
a_rscolq_round_stable_r003   9  8.1


 a_rscolq_round_stable_r003


,classwise_blend,min_server_alpha,residual_blend,round,rscolq_class_multipliers,rscolq_context_reason,rscolq_global_multiplier,rscolq_high,rscolq_low,rscolq_mean_class_multiplier,rscolq_mid,rscolq_model,rscolq_previous_round,server_anchor,variant
0,0.065,0.700000,0.140000,1,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",no_previous_stats,1.00,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",1.000000,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,0,13.0,a_rscolq_round_stable_r003
1,0.065,0.704444,0.131111,2,"[0.971859, 0.981843, 1.0, 1.0, 0.984673, 0.993...",previous_round_stats,0.15,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",0.984518,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,1,13.0,a_rscolq_round_stable_r003
2,0.065,0.708889,0.122222,3,"[1.0, 1.0, 1.0, 0.984331, 0.926893, 1.0, 1.0, ...",previous_round_stats,1.00,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",0.991122,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,2,13.0,a_rscolq_round_stable_r003
3,0.065,0.713333,0.113333,4,"[0.961213, 1.0, 1.0, 1.0, 0.991442, 1.0, 1.0, ...",previous_round_stats,0.15,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",0.989950,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,3,13.0,a_rscolq_round_stable_r003
4,0.065,0.717778,0.104444,5,"[1.0, 1.0, 1.0, 0.968379, 0.97823, 1.0, 1.0, 1...",previous_round_stats,1.00,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",0.994661,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,4,13.0,a_rscolq_round_stable_r003
5,0.065,0.722222,0.095556,6,"[0.976812, 1.0, 1.0, 1.0, 0.98719, 1.0, 1.0, 0...",previous_round_stats,0.15,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",0.989350,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,5,13.0,a_rscolq_round_stable_r003
6,0.065,0.726667,0.086667,7,"[1.0, 1.0, 1.0, 0.961436, 0.977186, 1.0, 1.0, ...",previous_round_stats,1.00,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",0.993862,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,6,13.0,a_rscolq_round_stable_r003
7,0.065,0.731111,0.077778,8,"[0.960593, 0.966193, 1.0, 0.981092, 0.985031, ...",previous_round_stats,0.15,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",0.980511,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,7,13.0,a_rscolq_round_stable_r003
8,0.065,0.735556,0.068889,9,"[1.0, 1.0, 1.0, 0.923506, 0.934738, 1.0, 1.0, ...",previous_round_stats,1.00,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",0.985339,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,8,13.0,a_rscolq_round_stable_r003
9,0.065,0.740000,0.060000,10,"[0.957155, 0.957669, 1.0, 0.962929, 0.940735, ...",previous_round_stats,0.15,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...",0.971894,"[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,9,13.0,a_rscolq_round_stable_r003


,run_name,variant,previous_round,previous_stats,global_multiplier,class_multipliers,mean_class_multiplier,reason,pseudo_inflation,quality_inflation
0,dqa_phase2_round001_client0_highway,a_rscolq_round_stable_r003,0,None,1.00,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",1.000000,no_previous_stats,NaN,NaN
1,dqa_phase2_round001_client1_citystreet,a_rscolq_round_stable_r003,0,None,1.00,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",1.000000,no_previous_stats,NaN,NaN
2,dqa_phase2_round001_client2_residential,a_rscolq_round_stable_r003,0,None,1.00,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",1.000000,no_previous_stats,NaN,NaN
3,dqa_phase2_round002_client0_highway,a_rscolq_round_stable_r003,1,"{'round': 1, 'path': '/app/Object_Detection/dy...",0.15,"[0.971859, 0.981843, 1.0, 1.0, 0.984673, 0.993...",0.984518,previous_round_stats,0.058301,0.196827
4,dqa_phase2_round002_client1_citystreet,a_rscolq_round_stable_r003,1,"{'round': 1, 'path': '/app/Object_Detection/dy...",0.15,"[0.971859, 0.981843, 1.0, 1.0, 0.984673, 0.993...",0.984518,previous_round_stats,0.058301,0.196827
5,dqa_phase2_round002_client2_residential,a_rscolq_round_stable_r003,1,"{'round': 1, 'path': '/app/Object_Detection/dy...",0.15,"[0.971859, 0.981843, 1.0, 1.0, 0.984673, 0.993...",0.984518,previous_round_stats,0.058301,0.196827
6,dqa_phase2_round003_client0_highway,a_rscolq_round_stable_r003,2,"{'round': 2, 'path': '/app/Object_Detection/dy...",1.00,"[1.0, 1.0, 1.0, 0.984331, 0.926893, 1.0, 1.0, ...",0.991122,previous_round_stats,0.000000,0.000000
7,dqa_phase2_round003_client1_citystreet,a_rscolq_round_stable_r003,2,"{'round': 2, 'path': '/app/Object_Detection/dy...",1.00,"[1.0, 1.0, 1.0, 0.984331, 0.926893, 1.0, 1.0, ...",0.991122,previous_round_stats,0.000000,0.000000
8,dqa_phase2_round003_client2_residential,a_rscolq_round_stable_r003,2,"{'round': 2, 'path': '/app/Object_Detection/dy...",1.00,"[1.0, 1.0, 1.0, 0.984331, 0.926893, 1.0, 1.0, ...",0.991122,previous_round_stats,0.000000,0.000000
9,dqa_phase2_round004_client0_highway,a_rscolq_round_stable_r003,3,"{'round': 3, 'path': '/app/Object_Detection/dy...",0.15,"[0.961213, 1.0, 1.0, 1.0, 0.991442, 1.0, 1.0, ...",0.989950,previous_round_stats,0.039560,0.190088


## 読み方

Aで見たいのは、08_3_6 Aの `r010` 劣化をR-SCoLQ multiplierが抑えられるか。

- `pseudo_total` が08_3_6ほど増えない、または増えても `global_multiplier` が下がって bbox/cls positive が絞られるなら設計は機能している。
- `r001/r002` のmAPが08_3_6と同程度で、`r010` の落ち込みが小さければ成功。
- `r001/r002` から既に悪い場合は、R-SCoLQ static model のscore分布がSCoLQより低すぎる可能性があるので、high thresholdを0.55付近に下げるか、B/Cではなくthreshold calibrationを先に見る。

In [9]:
import sys
sys.path.insert(0, "/app/Object_Detection")

from notebook_notify import notify_discord

notify_discord("０８＿３＿７終わった", title="実行終了")


DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)